# Notebook 5 — Real-time pipeline & Step 6 Capture results
| | |
|---|---|
| **Case Study** | 6 — Smart City IoT Data Platform (6.1, 6.3 Real-time pipelines, ML systems) |
| **Roll No** | `<ROLL_NO>` |
| **Name** | `<NAME>` |

## How to run the live pipeline (3 terminals, all in the project folder with `.venv` activated)

| Terminal | Command | What it does |
|---|---|---|
| 1 | `python -m src.streaming_job --reset` | starts 4 Spark Structured Streaming queries (wait for "4 streaming queries running") |
| 2 | `python -m src.replayer --reset --hours-per-tick 1 --interval 1` | sensors send 1 hour of readings per second |
| 3 | `streamlit run src/dashboard.py` | opens the live dashboard at http://localhost:8501 |

Let it run for 3–5 minutes (or until the replayer finishes), then stop terminals 1 and 2 with **Ctrl+C** and run this notebook.

```
replayer ──JSON files──► data/stream_input/{traffic,air}
                               │  readStream (file source, schema, maxFilesPerTrigger)
                               ▼
              ┌─────────── Spark Structured Streaming (trigger 5 s, watermark 2 h) ───────────┐
              │ traffic_predictions : features → saved model → next-hour forecast → alert     │
              │ air_predictions     : soft sensor → estimated CO → alert                      │
              │ traffic_window_6h   : 6-hour tumbling windows per junction                     │
              │ city_snapshot       : stream-stream join traffic ⨝ air (same hour)            │
              └──── foreachBatch → output/gold/stream/<query>/batch_id=N (idempotent) ─────────┘
                               │
                               ▼
                     Streamlit dashboard (refresh 5 s)
```

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from pyspark.sql import functions as F
from src import config
from src.spark_utils import get_spark
spark = get_spark("NB05-StreamingResults")
print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

In [ ]:
def table(name):
    return spark.read.parquet(config.p(config.GOLD / "stream" / name))

traffic = table("traffic_predictions"); air = table("air_predictions")
windows = table("traffic_window_6h");   snapshot = table("city_snapshot")
for name, df in [("traffic_predictions", traffic), ("air_predictions", air), ("traffic_window_6h", windows), ("city_snapshot", snapshot)]:
    print(f"{name:20s} rows={df.count():6d}  micro-batches={df.select('batch_id').distinct().count()}")

### Live forecasts compared with what actually happened one hour later

In [ ]:
actual = traffic.select("junction", F.col("event_time").alias("forecast_for"), F.col("vehicles").alias("actual_next"))
scored = traffic.join(actual, ["junction", "forecast_for"])
scored.groupBy("junction").agg(F.count("*").alias("forecasts"),
                               F.round(F.avg(F.abs(F.col("predicted_vehicles") - F.col("actual_next"))), 2).alias("live_MAE"),
                               F.sum(F.col("congestion_alert").cast("int")).alias("alerts")).orderBy("junction").show()
scored.select("junction", "event_time", "vehicles", "forecast_for", "predicted_vehicles", "actual_next", "congestion_alert") \
      .orderBy(F.desc("forecast_for")).show(8)

In [ ]:
print("Congestion alerts (latest)")
traffic.filter("congestion_alert").select("forecast_for", "junction", "predicted_vehicles", "alert_threshold").orderBy(F.desc("forecast_for")).show(10)
print("High-CO alerts (latest)")
air.filter("co_alert").select("event_time", "calibrated_co", "estimated_co", "co_gt", "temperature").orderBy(F.desc("event_time")).show(10)

### Soft sensor in the live stream: drift and online recalibration
Cheap gas sensors **drift** as they age, so the raw model estimate gets steadily worse over the months after training. Whenever the reference analyser is online we know the truth, so each micro-batch scales the estimate by **sum(true) / sum(estimated) over the last 24 readings** that had a reference value (only past readings are used; weighting by magnitude stops near-zero night readings from dominating). The tables show the error per week, raw vs corrected.

In [ ]:
air.select(F.count("*").alias("readings"),
           F.sum(F.col("co_gt").isNull().cast("int")).alias("analyser_offline"),
           F.round(F.avg(F.abs(F.col("estimated_co") - F.col("co_gt"))), 3).alias("MAE_raw"),
           F.round(F.avg(F.abs(F.col("calibrated_co") - F.col("co_gt"))), 3).alias("MAE_drift_corrected")).show()
air.filter(F.col("co_gt").isNotNull()).groupBy(F.date_trunc("week", "original_time").alias("week_of_original_data"))    .agg(F.round(F.avg(F.col("estimated_co") - F.col("co_gt")), 2).alias("bias_raw"),
        F.round(F.avg(F.col("calibrated_co") - F.col("co_gt")), 2).alias("bias_corrected"),
        F.count("*").alias("readings")).orderBy("week_of_original_data").show()

### Windowed aggregates (emitted only after the watermark passes) and the stream-stream join

In [ ]:
windows.orderBy("window_start", "junction").show(8, truncate=False)
snapshot.orderBy("event_time").show(5)

### Latency (ingest → result written)

In [ ]:
traffic.select(F.round(F.expr("percentile(latency_sec, 0.5)"), 2).alias("p50_s"),
               F.round(F.expr("percentile(latency_sec, 0.95)"), 2).alias("p95_s"),
               F.round(F.max("latency_sec"), 2).alias("max_s")).show()

> 📸 **Screenshot 7 (Step 6 results):** the tables above **and** the dashboard: Traffic tab, Air tab, 6-h windows tab, Pipeline-health tab. Also Spark UI → *Structured Streaming* tab (input rate vs process rate per query).

In [ ]:
spark.stop()